In [ ]:
# Stage 3.5: Post-hoc TTL Correction
import json
import re
from pathlib import Path
from datetime import datetime
from collections import Counter

# Paths
BASE_DIR = Path('/Users/umair/Synthesising Regulatory Ontologies')

# Input: Stage 2 manifest (UPDATE if folder name different)
STAGE2_MANIFEST = BASE_DIR / '3 - Extraction and  Validation Layer' / 'output' / 'stage2_extraction' / 'stage2_manifest.json'

# Output: corrected TTLs go here
STAGE3_5_OUTPUT = BASE_DIR / '3 - Extraction and  Validation Layer' / 'output' / 'stage3_5_corrected'
STAGE3_5_OUTPUT.mkdir(parents=True, exist_ok=True)
CORRECTED_MANIFEST = STAGE3_5_OUTPUT / 'stage3_5_manifest.json'

# Verify paths
assert STAGE2_MANIFEST.exists(), f'Stage 2 manifest not found: {STAGE2_MANIFEST}'

# Load Stage 2 data
with open(STAGE2_MANIFEST, encoding='utf-8') as f:
    stage2_data = json.load(f)

print(f'✓ Stage 2 chunks loaded: {len(stage2_data["results"])}')
print(f'✓ Output directory: {STAGE3_5_OUTPUT}')

In [ ]:
# Stage 3.5 Fix Functions - v3 with Fix 5 and Fix 6

def fix_1_remove_issues_from_norms(ttl_text):
    if 'cco:issues' not in ttl_text:
        return ttl_text, 0
    lines = ttl_text.split('\n')
    out_lines = []
    fixes_count = 0
    in_norm_block = False
    norm_pattern = re.compile(r'^\s*data:\S+\s+a\s+cco:(Obligation|Permission|Prohibition|Exception|Norm)\b')
    new_subject_pattern = re.compile(r'^\s*(data:\S+|gro\S*:\S+|cco:\S+)\s+a\s+')
    for line in lines:
        if norm_pattern.match(line):
            in_norm_block = True
            out_lines.append(line)
            continue
        if in_norm_block:
            if new_subject_pattern.match(line) and 'cco:issues' not in line:
                in_norm_block = False
        if in_norm_block and 'cco:issues' in line:
            fixes_count += 1
            if line.rstrip().endswith('.') and out_lines:
                if out_lines[-1].rstrip().endswith(';'):
                    out_lines[-1] = re.sub(r';\s*$', ' .', out_lines[-1])
            continue
        out_lines.append(line)
    return '\n'.join(out_lines), fixes_count


def fix_2_retype_exception_norms(ttl_text):
    if 'cco:Exception' not in ttl_text:
        return ttl_text, 0
    pattern = re.compile(r'(data:\S+\s+a\s+)cco:Exception\b')
    fixes_count = 0
    def replace_if_norm_like(match):
        nonlocal fixes_count
        # Only re-type _norm_ URIs typed as Exception, NOT _exception_ URIs
        instance_uri = match.group(1).strip().rstrip(' a')
        if '_exception_' in instance_uri:
            return match.group(0)  # Leave actual exception instances alone
        lookahead = ttl_text[match.end():match.end()+1000]
        next_decl = re.search(r'\n\s*(data:|gro\S*:|cco:)\S+\s+a\s+', lookahead)
        if next_decl:
            lookahead = lookahead[:next_decl.start()]
        has_norm_props = (
            'gro:hasSubject' in lookahead or
            'cco:appliesToRole' in lookahead or
            'cco:hasAction' in lookahead or
            'cco:hasObject' in lookahead
        )
        if has_norm_props:
            fixes_count += 1
            return match.group(1) + 'cco:Norm'
        return match.group(0)
    ttl_text = pattern.sub(replace_if_norm_like, ttl_text)
    return ttl_text, fixes_count


def fix_3_strip_hasEndTime_from_norms(ttl_text):
    if 'cco:hasEndTime' not in ttl_text:
        return ttl_text, 0
    lines = ttl_text.split('\n')
    out_lines = []
    fixes_count = 0
    in_norm_block = False
    norm_pattern = re.compile(r'^\s*data:\S+\s+a\s+cco:(Obligation|Permission|Prohibition|Exception|Norm)\b')
    new_subject_pattern = re.compile(r'^\s*(data:\S+|gro\S*:\S+|cco:\S+)\s+a\s+')
    for line in lines:
        if norm_pattern.match(line):
            in_norm_block = True
            out_lines.append(line)
            continue
        if in_norm_block:
            if new_subject_pattern.match(line):
                in_norm_block = False
        if in_norm_block and 'cco:hasEndTime' in line:
            fixes_count += 1
            if line.rstrip().endswith('.') and out_lines:
                if out_lines[-1].rstrip().endswith(';'):
                    out_lines[-1] = re.sub(r';\s*$', ' .', out_lines[-1])
            continue
        out_lines.append(line)
    return '\n'.join(out_lines), fixes_count


def fix_4_strip_invalid_date_literals(ttl_text):
    if '^^xsd:date' not in ttl_text and '^^xsd:gMonthDay' not in ttl_text:
        return ttl_text, 0
    valid_date = re.compile(r'^-?\d{4}-\d{2}-\d{2}$')
    valid_gmonth = re.compile(r'^--\d{2}-\d{2}$')
    fixes_count = 0
    def check_and_fix(match):
        nonlocal fixes_count
        value = match.group(1)
        dtype = match.group(2)
        is_valid = False
        if dtype == 'date' and valid_date.match(value):
            is_valid = True
        elif dtype == 'gMonthDay' and valid_gmonth.match(value):
            is_valid = True
        if is_valid:
            return match.group(0)
        else:
            fixes_count += 1
            return f'"{value}"'
    pattern = re.compile(r'"([^"]*)"\s*\^\^xsd:(date|gMonthDay)')
    ttl_text = pattern.sub(check_and_fix, ttl_text)
    return ttl_text, fixes_count


def fix_5_convert_hasCondition_on_norms(ttl_text):
    """Convert cco:hasCondition → cco:appliesUnder on Obligation/Permission/Prohibition/Norm.
    Exception instances legitimately use cco:hasCondition — left alone."""
    if 'cco:hasCondition' not in ttl_text:
        return ttl_text, 0
    
    lines = ttl_text.split('\n')
    out_lines = []
    fixes_count = 0
    in_norm_block = False
    # Exclude Exception from norm_pattern here — Exception legitimately uses hasCondition
    norm_pattern = re.compile(r'^\s*data:\S+\s+a\s+cco:(Obligation|Permission|Prohibition|Norm)\b')
    new_subject_pattern = re.compile(r'^\s*(data:\S+|gro\S*:\S+|cco:\S+)\s+a\s+')
    
    for line in lines:
        if norm_pattern.match(line):
            in_norm_block = True
            out_lines.append(line)
            continue
        if in_norm_block:
            if new_subject_pattern.match(line):
                in_norm_block = False
        
        if in_norm_block and 'cco:hasCondition' in line and 'cco:hasConditionExpression' not in line:
            new_line = line.replace('cco:hasCondition', 'cco:appliesUnder')
            out_lines.append(new_line)
            fixes_count += 1
            continue
        out_lines.append(line)
    
    return '\n'.join(out_lines), fixes_count


def fix_6_add_exception_links(ttl_text):
    """Add cco:modifiesNorm + cco:hasCondition to Exception instances
    based on reverse links from Norms (cco:hasException references Exception)."""
    if 'cco:Exception' not in ttl_text or 'cco:hasException' not in ttl_text:
        return ttl_text, 0
    
    # Find all Exception instances
    exception_pattern = re.compile(r'^\s*(data:\S+)\s+a\s+cco:Exception\b', re.MULTILINE)
    exceptions = set(m.group(1) for m in exception_pattern.finditer(ttl_text))
    
    if not exceptions:
        return ttl_text, 0
    
    # Find Norm → Exception links (cco:hasException references)
    # Pattern: data:X_norm_Y ... cco:hasException data:X_exception_Z
    norm_exception_pattern = re.compile(
        r'(data:\S+)\s+a\s+cco:(Obligation|Permission|Prohibition|Norm)\b[^.]*?cco:hasException\s+(data:\S+)',
        re.DOTALL
    )
    
    # Find Condition instances in chunk (for hasCondition linking)
    condition_pattern = re.compile(r'^\s*(data:\S+)\s+a\s+cco:Condition\b', re.MULTILINE)
    conditions = list(condition_pattern.finditer(ttl_text))
    first_condition = conditions[0].group(1) if conditions else None
    
    # Build mapping: exception → norm
    exception_to_norm = {}
    for m in norm_exception_pattern.finditer(ttl_text):
        norm_uri = m.group(1)
        exc_uri = m.group(3)
        if exc_uri in exceptions:
            exception_to_norm[exc_uri] = norm_uri
    
    if not exception_to_norm:
        return ttl_text, 0
    
    # For each Exception, check if it has modifiesNorm and hasCondition; if not, add them
    fixes_count = 0
    lines = ttl_text.split('\n')
    out_lines = []
    
    i = 0
    while i < len(lines):
        line = lines[i]
        out_lines.append(line)
        
        # Check if this line starts an Exception block
        match = re.match(r'^\s*(data:\S+)\s+a\s+cco:Exception\b', line)
        if match:
            exc_uri = match.group(1)
            if exc_uri in exception_to_norm:
                # Collect Exception block until next instance declaration
                block_end = i + 1
                while block_end < len(lines):
                    if re.match(r'^\s*(data:\S+|gro\S*:\S+|cco:\S+)\s+a\s+', lines[block_end]):
                        break
                    block_end += 1
                
                # Extract block content
                block_text = '\n'.join(lines[i:block_end])
                has_modifiesNorm = 'cco:modifiesNorm' in block_text
                has_hasCondition = 'cco:hasCondition' in block_text and 'cco:hasConditionExpression' not in block_text
                
                if not has_modifiesNorm or (not has_hasCondition and first_condition):
                    # Find last property line in block (ends with .)
                    # Need to convert its '.' to ';' and add new properties
                    last_prop_idx = -1
                    for j in range(block_end - 1, i, -1):
                        if lines[j].rstrip().endswith('.'):
                            last_prop_idx = j
                            break
                    
                    if last_prop_idx > 0:
                        # Convert last . to ;
                        out_lines.pop()  # We already added i, will re-add up to last
                        for k in range(i, last_prop_idx):
                            out_lines.append(lines[k])
                        # Convert the last . to ;
                        out_lines.append(re.sub(r'\.\s*$', ' ;', lines[last_prop_idx]))
                        
                        # Add missing properties
                        indent = '    '
                        if not has_modifiesNorm:
                            out_lines.append(f'{indent}cco:modifiesNorm {exception_to_norm[exc_uri]} ;')
                            fixes_count += 1
                        if not has_hasCondition and first_condition:
                            out_lines.append(f'{indent}cco:hasCondition {first_condition} ;')
                            fixes_count += 1
                        
                        # Replace last ; with .
                        out_lines[-1] = re.sub(r';\s*$', ' .', out_lines[-1])
                        
                        i = last_prop_idx + 1
                        continue
        
        i += 1
    
    return '\n'.join(out_lines), fixes_count


def apply_all_fixes(ttl_text):
    fix_counts = {
        'fix_1_issues': 0,
        'fix_2_exception_retype': 0,
        'fix_3_hasEndTime': 0,
        'fix_4_invalid_dates': 0,
        'fix_5_hasCondition_on_norm': 0,
        'fix_6_exception_links': 0,
    }
    ttl_text, c1 = fix_1_remove_issues_from_norms(ttl_text)
    fix_counts['fix_1_issues'] = c1
    ttl_text, c2 = fix_2_retype_exception_norms(ttl_text)
    fix_counts['fix_2_exception_retype'] = c2
    ttl_text, c3 = fix_3_strip_hasEndTime_from_norms(ttl_text)
    fix_counts['fix_3_hasEndTime'] = c3
    ttl_text, c4 = fix_4_strip_invalid_date_literals(ttl_text)
    fix_counts['fix_4_invalid_dates'] = c4
    ttl_text, c5 = fix_5_convert_hasCondition_on_norms(ttl_text)
    fix_counts['fix_5_hasCondition_on_norm'] = c5
    ttl_text, c6 = fix_6_add_exception_links(ttl_text)
    fix_counts['fix_6_exception_links'] = c6
    return ttl_text, fix_counts


print(' All 6 fix functions loaded successfully.')

In [ ]:
# Apply fixes to all chunks
print('=' * 70)
print('STAGE 3.5: APPLYING POST-HOC FIXES')
print('=' * 70)

corrected_results = []
total_fix_counts = Counter()

for i, s2r in enumerate(stage2_data['results'], 1):
    uid = s2r['unit_id']
    original_ttl = s2r.get('turtle_text', '')
    
    if not original_ttl:
        corrected_results.append({
            'unit_id': uid,
            'jurisdiction': s2r['jurisdiction'],
            'status': 'skipped_empty',
            'fix_counts': {},
        })
        continue
    
    corrected_ttl, fix_counts = apply_all_fixes(original_ttl)
    
    total_fixes_this_chunk = sum(fix_counts.values())
    for fix_name, count in fix_counts.items():
        total_fix_counts[fix_name] += count
    
    corrected_results.append({
        'unit_id': uid,
        'jurisdiction': s2r['jurisdiction'],
        'status': 'corrected' if total_fixes_this_chunk > 0 else 'no_changes',
        'fix_counts': fix_counts,
    })
    
    ttl_path = STAGE3_5_OUTPUT / f'{uid}.ttl'
    ttl_path.write_text(corrected_ttl, encoding='utf-8')
    
    if i % 200 == 0:
        print(f'  Processed {i}/{len(stage2_data["results"])} chunks')

print(f'\n=== Fix Application Summary ===')
print(f'Total chunks processed           : {len(corrected_results)}')
print(f'Chunks corrected (≥1 fix)        : {sum(1 for r in corrected_results if r.get("fix_counts") and sum(r["fix_counts"].values()) > 0)}')
print(f'\nFix 1 (cco:issues removed)         : {total_fix_counts["fix_1_issues"]}')
print(f'Fix 2 (Exception → Norm retype)    : {total_fix_counts["fix_2_exception_retype"]}')
print(f'Fix 3 (cco:hasEndTime stripped)    : {total_fix_counts["fix_3_hasEndTime"]}')
print(f'Fix 4 (invalid dates stripped)     : {total_fix_counts["fix_4_invalid_dates"]}')
print(f'Fix 5 (hasCondition→appliesUnder)  : {total_fix_counts["fix_5_hasCondition_on_norm"]}')
print(f'Fix 6 (exception links added)      : {total_fix_counts["fix_6_exception_links"]}')
print(f'TOTAL FIXES                        : {sum(total_fix_counts.values())}')

In [ ]:
# Save Stage 3.5 manifest
manifest_payload = {
    'metadata': {
        'stage': 'stage3_5_post_hoc_correction',
        'created_at': datetime.now().isoformat(),
        'source_manifest': str(STAGE2_MANIFEST),
        'fixes_applied': [
            'Fix 1: Remove cco:issues from Norm instances',
            'Fix 2: Re-type cco:Exception norms to cco:Norm',
            'Fix 3: Strip cco:hasEndTime from Norms',
            'Fix 4: Strip invalid xsd:date / xsd:gMonthDay literals',
        ],
    },
    'fix_counts_total': dict(total_fix_counts),
    'total_chunks': len(corrected_results),
    'results': corrected_results,
}

with open(CORRECTED_MANIFEST, 'w', encoding='utf-8') as f:
    json.dump(manifest_payload, f, indent=2, ensure_ascii=False)

print(f'✓ Manifest saved: {CORRECTED_MANIFEST}')
print(f'✓ Total TTL files in output: {sum(1 for f in STAGE3_5_OUTPUT.glob("*.ttl"))}')

In [ ]:
# Re-validate corrected TTLs with SHACL
from rdflib import Graph, RDF, URIRef
from pyshacl import validate

print('=' * 70)
print('STAGE 3.5: RE-VALIDATION WITH SHACL')
print('=' * 70)

SHACL_SHAPES_PATH = BASE_DIR / '1 - Foundation Layer' / 'cco_shapes.ttl'
CCO_ONTOLOGY = BASE_DIR / '1 - Foundation Layer' / 'CCO.ttl'

cco_graph = Graph()
cco_graph.parse(str(CCO_ONTOLOGY), format='turtle')

shacl_graph = Graph()
shacl_graph.parse(str(SHACL_SHAPES_PATH), format='turtle')

print(f'CCO triples: {len(cco_graph)}')
print(f'SHACL triples: {len(shacl_graph)}')

SH_VIOLATION = URIRef('http://www.w3.org/ns/shacl#Violation')
SH_WARNING = URIRef('http://www.w3.org/ns/shacl#Warning')
SH_SEVERITY = URIRef('http://www.w3.org/ns/shacl#resultSeverity')
SH_VALIDATIONRESULT = URIRef('http://www.w3.org/ns/shacl#ValidationResult')

revalidation_results = []
n_pass = 0
n_fail = 0
total_violations = 0
total_warnings = 0

print(f'\nValidating {len(corrected_results)} chunks...')

for i, cr in enumerate(corrected_results, 1):
    uid = cr['unit_id']
    jur = cr['jurisdiction']
    
    ttl_path = STAGE3_5_OUTPUT / f'{uid}.ttl'
    if not ttl_path.exists():
        revalidation_results.append({
            'unit_id': uid, 'jurisdiction': jur,
            'pass': False, 'n_violations': -1,
            'skip_reason': 'TTL not found'
        })
        n_fail += 1
        continue
    
    ttl = ttl_path.read_text(encoding='utf-8')
    if not ttl.strip():
        revalidation_results.append({
            'unit_id': uid, 'jurisdiction': jur,
            'pass': False, 'n_violations': -1,
            'skip_reason': 'Empty TTL'
        })
        n_fail += 1
        continue
    
    try:
        data_graph = Graph()
        data_graph.parse(data=ttl, format='turtle')
        combined = data_graph + cco_graph
        
        conforms, results_graph, _ = validate(
            combined,
            shacl_graph=shacl_graph,
            inference='rdfs',
            allow_warnings=True,
            abort_on_first=False,
        )
        
        n_v = 0
        n_w = 0
        for vr in results_graph.subjects(RDF.type, SH_VALIDATIONRESULT):
            sev = next(results_graph.objects(vr, SH_SEVERITY), None)
            if sev == SH_VIOLATION: n_v += 1
            elif sev == SH_WARNING: n_w += 1
        
        passed = (n_v == 0)
        revalidation_results.append({
            'unit_id': uid, 'jurisdiction': jur,
            'pass': passed,
            'n_violations': n_v,
            'n_warnings': n_w,
        })
        
        total_violations += n_v
        total_warnings += n_w
        
        if passed: n_pass += 1
        else: n_fail += 1
        
        if i % 100 == 0:
            print(f'  Validated {i}/{len(corrected_results)} | Pass: {n_pass}, Fail: {n_fail}')
            
    except Exception as e:
        revalidation_results.append({
            'unit_id': uid, 'jurisdiction': jur,
            'pass': False, 'n_violations': -1,
            'error': str(e)[:100]
        })
        n_fail += 1

print(f'\n{"=" * 70}')
print(f'RE-VALIDATION COMPLETE')
print(f'{"=" * 70}')
print(f'Total chunks:       {len(revalidation_results)}')
print(f'Passed (V=0):       {n_pass} ({n_pass/len(revalidation_results)*100:.1f}%)')
print(f'Failed (V>0):       {n_fail} ({n_fail/len(revalidation_results)*100:.1f}%)')
print(f'Total violations:   {total_violations}')
print(f'Total warnings:     {total_warnings}')

print(f'\n=== Improvement vs Stage 3 ===')
print(f'Stage 3 pass rate:   81.9%')
print(f'Stage 3.5 pass rate: {n_pass/len(revalidation_results)*100:.1f}%')
improvement = (n_pass/len(revalidation_results)*100) - 81.9
print(f'Improvement:         {improvement:+.1f} percentage points')

In [ ]:
# Save final revalidation manifest
final_manifest = {
    'metadata': {
        'stage': 'stage3_5_revalidation',
        'created_at': datetime.now().isoformat(),
        'corrected_ttls_from': str(STAGE3_5_OUTPUT),
        'shacl_shapes': str(SHACL_SHAPES_PATH),
        'cco_ontology': str(CCO_ONTOLOGY),
    },
    'summary': {
        'total_chunks': len(revalidation_results),
        'passed_chunks': n_pass,
        'failed_chunks': n_fail,
        'pass_rate': n_pass / len(revalidation_results) if revalidation_results else 0,
        'total_violations': total_violations,
        'total_warnings': total_warnings,
        'stage3_pass_rate': 0.819,
        'improvement_pp': improvement,
    },
    'fix_application_summary': dict(total_fix_counts),
    'results': revalidation_results,
}

final_path = STAGE3_5_OUTPUT / 'stage3_5_revalidation_manifest.json'
with open(final_path, 'w', encoding='utf-8') as f:
    json.dump(final_manifest, f, indent=2, ensure_ascii=False)

print(f'✓ Final manifest saved: {final_path}')
print(f'\n🎯 Final Stage 3.5 Pass Rate: {n_pass/len(revalidation_results)*100:.1f}%')

In [ ]:
# Inspect 3 still-failing chunks to find remaining issue patterns
still_failing = [r for r in revalidation_results if not r['pass']]

print(f'Total still failing: {len(still_failing)}\n')

for i, fr in enumerate(still_failing[:3]):
    target_uid = fr['unit_id']
    print(f'=== {i+1}. {target_uid} | V={fr["n_violations"]} ===\n')
    
    # Show corrected TTL
    ttl_path = STAGE3_5_OUTPUT / f'{target_uid}.ttl'
    
    # Run SHACL on this one
    from rdflib import Graph
    from pyshacl import validate
    
    data_graph = Graph()
    data_graph.parse(ttl_path, format='turtle')
    combined = data_graph + cco_graph
    
    conforms, results_graph, results_text = validate(
        combined,
        shacl_graph=shacl_graph,
        inference='rdfs',
        allow_warnings=True,
        abort_on_first=False,
    )
    
    print('--- VIOLATION MESSAGES ---')
    print(results_text[:2000])
    print('\n' + '=' * 70 + '\n')

In [ ]:
# Check what's in corrected AUS-UNIT-00033
target_uid = 'AUS-UNIT-00033'
ttl_path = STAGE3_5_OUTPUT / f'{target_uid}.ttl'
content = ttl_path.read_text()

print('=== AUS-UNIT-00033 CORRECTED TTL ===\n')
print(content)
print('\n=== Does cco:Exception still appear? ===')
print('cco:Exception' in content)

# Check if fix 2 should have triggered
import re
pattern = re.compile(r'(data:\S+\s+a\s+)cco:Exception\b')
matches = list(pattern.finditer(content))
print(f'\nPattern matches found: {len(matches)}')
for m in matches:
    print(f'  Match: {m.group(0)}')

In [ ]:
# Inspect AUS-UNIT-00035 to see _exception_ structure
ttl_path = STAGE3_5_OUTPUT / 'AUS-UNIT-00035.ttl'
print(ttl_path.read_text())

In [ ]:
ttl_path = STAGE3_5_OUTPUT / 'AUS-UNIT-00035.ttl'
print(ttl_path.read_text())